In [2]:
import pandas as pd
import numpy as np

CLEANED = "../data/cleaned/"

# Load the joined dataset from Day 5
df = pd.read_csv(CLEANED + "ecommerce_joined.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'review_answer_timestamp',
    'shipping_limit_date'
])

print(f"Loaded: {len(df):,} rows × {df.shape[1]} columns")
print(f"Unique orders: {df['order_id'].nunique():,}")
print(f"\nColumn list:")
for col in df.columns:
    print(f"  - {col}")

Loaded: 110,189 rows × 25 columns
Unique orders: 96,470

Column list:
  - order_id
  - customer_id
  - order_status
  - order_purchase_timestamp
  - order_approved_at
  - order_delivered_carrier_date
  - order_delivered_customer_date
  - order_estimated_delivery_date
  - customer_unique_id
  - customer_zip_code_prefix
  - customer_city
  - customer_state
  - total_payment_value
  - payment_installments
  - dominant_payment_type
  - review_score
  - review_answer_timestamp
  - order_item_id
  - product_id
  - seller_id
  - shipping_limit_date
  - price
  - freight_value
  - product_category_name
  - product_category_name_english


In [3]:
# FEATURE 1: delivery_days
# How many days the customer waited from purchase to delivery
df['delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

# FEATURE 2: estimated_delay_days
# Positive = arrived LATE, Negative = arrived EARLY
df['estimated_delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

print("=== DELIVERY DAYS ===")
print(df['delivery_days'].describe().round(2))

print("\n=== ESTIMATED DELAY DAYS ===")
print(df['estimated_delay_days'].describe().round(2))

# Sanity check — no negative delivery days
neg_delivery = (df['delivery_days'] < 0).sum()
print(f"\nNegative delivery_days (should be 0): {neg_delivery}")

=== DELIVERY DAYS ===
count    110189.00
mean         12.01
std           9.45
min           0.00
25%           6.00
50%          10.00
75%          15.00
max         209.00
Name: delivery_days, dtype: float64

=== ESTIMATED DELAY DAYS ===
count    110189.00
mean        -12.03
std          10.16
min        -147.00
25%         -17.00
50%         -13.00
75%          -7.00
max         188.00
Name: estimated_delay_days, dtype: float64

Negative delivery_days (should be 0): 0


In [4]:
# FEATURE 3: is_late_delivery
# Yes if order arrived after the estimated delivery date
df['is_late_delivery'] = df['estimated_delay_days'].apply(
    lambda x: 'Yes' if x > 0 else 'No'
)

print("=== IS LATE DELIVERY ===")
print(df['is_late_delivery'].value_counts())
print(f"\nLate delivery rate: {(df['is_late_delivery'] == 'Yes').mean() * 100:.1f}%")

=== IS LATE DELIVERY ===
is_late_delivery
No     102925
Yes      7264
Name: count, dtype: int64

Late delivery rate: 6.6%


In [5]:
# FEATURE 4: order_month (e.g. 2017-11 for November 2017)
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# FEATURE 5: order_year
df['order_year'] = df['order_purchase_timestamp'].dt.year

# FEATURE 6: order_weekday (0=Monday, 6=Sunday)
df['order_weekday'] = df['order_purchase_timestamp'].dt.day_name()

# FEATURE 7: order_hour
df['order_hour'] = df['order_purchase_timestamp'].dt.hour

print("=== ORDER MONTH (sample) ===")
print(df['order_month'].value_counts().sort_index().head(10))

print("\n=== ORDER WEEKDAY ===")
print(df['order_weekday'].value_counts())

print("\n=== ORDER HOUR (distribution) ===")
print(df['order_hour'].value_counts().sort_index())

=== ORDER MONTH (sample) ===
order_month
2016-09       3
2016-10     313
2016-12       1
2017-01     913
2017-02    1858
2017-03    2897
2017-04    2569
2017-05    4003
2017-06    3489
2017-07    4416
Name: count, dtype: int64

=== ORDER WEEKDAY ===
order_weekday
Monday       17973
Tuesday      17857
Wednesday    17217
Thursday     16433
Friday       15697
Sunday       13127
Saturday     11885
Name: count, dtype: int64

=== ORDER HOUR (distribution) ===
order_hour
0     2655
1     1264
2      575
3      299
4      242
5      213
6      532
7     1343
8     3337
9     5346
10    6881
11    7272
12    6660
13    7224
14    7415
15    7214
16    7478
17    6837
18    6390
19    6577
20    6733
21    6756
22    6398
23    4548
Name: count, dtype: int64


In [6]:
# FEATURE 8: revenue
# IMPORTANT: revenue = price + freight_value
# This definition must stay consistent across Python, SQL, Tableau, Flask, frontend
df['revenue'] = df['price'] + df['freight_value']

# FEATURE 9: freight_ratio
# How much of the item cost is just shipping
# Only calculate where price > 0 to avoid division by zero
df['freight_ratio'] = df.apply(
    lambda row: round(row['freight_value'] / row['price'], 4)
    if row['price'] > 0 else None,
    axis=1
)

print("=== REVENUE ===")
print(df['revenue'].describe().round(2))
print(f"\nTotal revenue across all items: R$ {df['revenue'].sum():,.2f}")

print("\n=== FREIGHT RATIO ===")
print(df['freight_ratio'].describe().round(4))
print(f"\nRows where freight > price (ratio > 1): {(df['freight_ratio'] > 1).sum():,}")

=== REVENUE ===
count    110189.00
mean        139.93
std         189.32
min           6.08
25%          55.18
50%          92.12
75%         157.47
max        6929.31
Name: revenue, dtype: float64

Total revenue across all items: R$ 15,418,394.83

=== FREIGHT RATIO ===
count    110189.0000
mean          0.3208
std           0.3478
min           0.0000
25%           0.1345
50%           0.2318
75%           0.3932
max          26.2353
Name: freight_ratio, dtype: float64

Rows where freight > price (ratio > 1): 4,008


In [7]:
# FEATURE 10: review_group
# Makes satisfaction easier to visualize in charts
def assign_review_group(score):
    if pd.isna(score):
        return 'No Review'
    elif score <= 2:
        return 'Low'
    elif score == 3:
        return 'Medium'
    else:
        return 'High'

df['review_group'] = df['review_score'].apply(assign_review_group)

print("=== REVIEW GROUP ===")
print(df['review_group'].value_counts())
print(f"\nDistribution %:")
print((df['review_group'].value_counts() / len(df) * 100).round(1))

=== REVIEW GROUP ===
review_group
High         84011
Low          16156
Medium        9195
No Review      827
Name: count, dtype: int64

Distribution %:
review_group
High         76.2
Low          14.7
Medium        8.3
No Review     0.8
Name: count, dtype: float64


In [8]:
# FEATURE 11: delivery_bucket
# Groups delivery speed into Fast / Normal / Slow
# Thresholds based on the delivery_days distribution from Cell 2
# Fast:   <= 7 days
# Normal: 8 to 20 days
# Slow:   > 20 days

def assign_delivery_bucket(days):
    if pd.isna(days):
        return 'Unknown'
    elif days <= 7:
        return 'Fast'
    elif days <= 20:
        return 'Normal'
    else:
        return 'Slow'

df['delivery_bucket'] = df['delivery_days'].apply(assign_delivery_bucket)

print("=== DELIVERY BUCKET ===")
print(df['delivery_bucket'].value_counts())
print(f"\nDistribution %:")
print((df['delivery_bucket'].value_counts() / len(df) * 100).round(1))

=== DELIVERY BUCKET ===
delivery_bucket
Normal    57502
Fast      38720
Slow      13967
Name: count, dtype: int64

Distribution %:
delivery_bucket
Normal    52.2
Fast      35.1
Slow      12.7
Name: count, dtype: float64


In [9]:
# Confirm all engineered features exist and have correct types
feature_cols = [
    'delivery_days', 'estimated_delay_days', 'is_late_delivery',
    'order_month', 'order_year', 'order_weekday', 'order_hour',
    'revenue', 'freight_ratio', 'review_group', 'delivery_bucket'
]

print("=== FEATURE SUMMARY ===")
for col in feature_cols:
    null_count = df[col].isnull().sum()
    dtype = df[col].dtype
    print(f"  {col:<25} dtype: {str(dtype):<10} nulls: {null_count:,}")

print(f"\nTotal columns in dataset: {df.shape[1]}")
print(f"Total rows:               {len(df):,}")

=== FEATURE SUMMARY ===
  delivery_days             dtype: int64      nulls: 0
  estimated_delay_days      dtype: int64      nulls: 0
  is_late_delivery          dtype: object     nulls: 0
  order_month               dtype: object     nulls: 0
  order_year                dtype: int32      nulls: 0
  order_weekday             dtype: object     nulls: 0
  order_hour                dtype: int32      nulls: 0
  revenue                   dtype: float64    nulls: 0
  freight_ratio             dtype: float64    nulls: 0
  review_group              dtype: object     nulls: 0
  delivery_bucket           dtype: object     nulls: 0

Total columns in dataset: 36
Total rows:               110,189


In [10]:
# These numbers will become your KPIs in SQL, Flask, and the frontend
# Write them down — they should match exactly in every tool later

total_orders   = df['order_id'].nunique()
total_revenue  = df['revenue'].sum()
avg_delivery   = df.drop_duplicates('order_id')['delivery_days'].mean()
late_rate      = (df.drop_duplicates('order_id')['is_late_delivery'] == 'Yes').mean() * 100
avg_review     = df.drop_duplicates('order_id')['review_score'].mean()

print("=" * 45)
print("  DASHBOARD KPIs — RECORD THESE NUMBERS")
print("=" * 45)
print(f"  Total Orders:        {total_orders:,}")
print(f"  Total Revenue:       R$ {total_revenue:,.2f}")
print(f"  Avg Delivery Days:   {avg_delivery:.2f}")
print(f"  Late Delivery Rate:  {late_rate:.2f}%")
print(f"  Avg Review Score:    {avg_review:.2f}")
print("=" * 45)
print("\n⚠️  Save these numbers. SQL, Flask, and Tableau must match them.")

  DASHBOARD KPIs — RECORD THESE NUMBERS
  Total Orders:        96,470
  Total Revenue:       R$ 15,418,394.83
  Avg Delivery Days:   12.09
  Late Delivery Rate:  6.77%
  Avg Review Score:    4.16

⚠️  Save these numbers. SQL, Flask, and Tableau must match them.


In [11]:
# ── Round financial columns before export (prevents MySQL truncation warnings) ──
cols_to_round = ['price', 'freight_value', 'revenue', 'freight_ratio', 'total_payment_value']
for col in cols_to_round:
    if col in df.columns:
        df[col] = df[col].round(4)

# ── Save CSV ──
output_path = CLEANED + "ecommerce_orders_features.csv"
df.to_csv(output_path, index=False)

print(f"✅ Feature-rich dataset saved: {output_path}")
print(f"   Rows:    {len(df):,}")
print(f"   Columns: {df.shape[1]}")
print(f"\nFull column list:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

✅ Feature-rich dataset saved: ../data/cleaned/ecommerce_orders_features.csv
   Rows:    110,189
   Columns: 36

Full column list:
   1. order_id
   2. customer_id
   3. order_status
   4. order_purchase_timestamp
   5. order_approved_at
   6. order_delivered_carrier_date
   7. order_delivered_customer_date
   8. order_estimated_delivery_date
   9. customer_unique_id
  10. customer_zip_code_prefix
  11. customer_city
  12. customer_state
  13. total_payment_value
  14. payment_installments
  15. dominant_payment_type
  16. review_score
  17. review_answer_timestamp
  18. order_item_id
  19. product_id
  20. seller_id
  21. shipping_limit_date
  22. price
  23. freight_value
  24. product_category_name
  25. product_category_name_english
  26. delivery_days
  27. estimated_delay_days
  28. is_late_delivery
  29. order_month
  30. order_year
  31. order_weekday
  32. order_hour
  33. revenue
  34. freight_ratio
  35. review_group
  36. delivery_bucket
